# Text classification with Hugging Face

In [1]:
from transformers import pipeline
from sklearn.metrics import accuracy_score

## Hugging Face pipelines for sentiment analysis

### Analyzing the sentiment of a review

In [2]:
# Define the sentiment analysis pipeline
classifier = pipeline(   
    task="sentiment-analysis", 
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

review_text = "The new update made the app much faster and easier to use!"

# Get sentiment prediction
result = classifier(review_text)

print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9979780316352844}]


### Batch classifying multiple reviews

In [3]:
classifier = pipeline(
  task="sentiment-analysis",
  model="distilbert-base-uncased-finetuned-sst-2-english"
)

review_batch = [
    "Absolutely love the new design!",
    "The app crashes every time I open it.",
    "Customer support was helpful and quick.",
    "Too many ads make it unusable.",
    "Everything works fine, but it’s a bit slow."
]

# Classify sentiments
results = classifier(review_batch)
print(results)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998812675476074}, {'label': 'NEGATIVE', 'score': 0.9995560050010681}, {'label': 'POSITIVE', 'score': 0.9994823932647705}, {'label': 'NEGATIVE', 'score': 0.9997507929801941}, {'label': 'NEGATIVE', 'score': 0.9972020387649536}]


### Comparing models in labelled review data

In [4]:
# Load sentiment analysis models
pipe_a = pipeline(task="sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
#pipe_b = pipeline(task="sentiment-analysis", model="abilfad/sentiment-binary-dicoding")

texts = ["I can't believe how great this app is, it’s honestly a game-changer!", 'It’s not as bad as I expected, but still not great.', 'The update did fix a lot of issues, so I’m glad they listened to feedback.', 'It’s just another buggy version with more problems than before.', "I'm amazed by how quickly it works now, truly impressed.", "Oh look, the app works for once. Not sure if it's a fluke or progress.", "The new design is a pleasant surprise — I didn't expect it to be this good.", 'The app keeps freezing when I need it most. Terrible experience.', 'I didn’t think it was possible, but this update made things even worse.', 'Wow, finally an update that makes sense. So happy about this!', 'At least the app doesn’t crash every five minutes now. That’s something.', 'I’ve never been so impressed by an app’s performance. Outstanding!', 'I’m sure it works for some people, but not for me. A lot of problems.', 'It’s almost like the developers actually care now. Big improvement!', 'The app is still the same: slow, frustrating, and useless.', 'The new features are cool, but the app is still very glitchy.', 'I wish I had known about this app sooner, it really does make a difference.', 'I regret wasting my time on this app. I’ve had enough.', 'I’m so happy they fixed that bug, I didn’t realize how much it bothered me.', 'The app does some things right, but most of it is just annoying.']

# Generate predictions
preds_a = [res['label'] for res in pipe_a(texts)]
#preds_b = [res['label'] for res in pipe_b(texts)]

true_labels = ['POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE']

# Evaluate accuracies
acc_a = accuracy_score(preds_a, true_labels)
#acc_b = accuracy_score(preds_b, true_labels)
print(f"Accuracy - Model A: {acc_a:.2f}")
#print(f"Accuracy - Model B: {acc_b:.2f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Accuracy - Model A: 0.90


## Zero-shot classification and QNLI

### Zero-shot classification for support tickets

In [5]:
# Initialize the zero-shot classifier
classifier = pipeline(
    task="zero-shot-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
)

ticket_text = "I was charged twice for my subscription this month. Can you please refund the extra charge?"
candidate_labels = ["Billing", "Technical Issue", "Account Access"]

# Classify the ticket
result = classifier(ticket_text, candidate_labels)

print(result['labels'])
print(result['scores'])

/home/jorge/.pyenv/versions/natural-language-processing-in-python/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

['Billing', 'Account Access', 'Technical Issue']
[0.9698624014854431, 0.016557542607188225, 0.013580020517110825]


### Does the text answer the question?

In [6]:
# Initialize the QNLI pipeline
classifier = pipeline(
    task="text-classification",
    model="cross-encoder/qnli-electra-base"
)

passage = "Our refund policy allows customers to return any item within 30 days of purchase, provided the item is in its original condition and accompanied by the receipt. Refunds are issued to the original payment method within 5–7 business days."
question = "Can I get a refund if I return a product after 20 days?"

# Get the result
result = classifier({"text": question, "text_pair": passage})
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'label': 'LABEL_0', 'score': 0.9747660756111145}


## Question similarity and grammatical correctness

### Detecting duplicate questions

In [7]:
# Initialize the pipeline
classifier = pipeline(task="text-classification", model="textattack/distilbert-base-uncased-QQP")

question_1 = "What's the process to change my password?"
question_2 = "How do I reset my account password?"

# Detect if the two questions are paraphrases
result = classifier({
    "text": question_1,
    "text_pair": question_2
})

print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'label': 'LABEL_1', 'score': 0.5704372525215149}


### Checking grammatical correctness

In [8]:
# Initialize the pipeline
classifier = pipeline(
    task="text-classification",
    model="textattack/bert-base-uncased-CoLA"
)

user_text = "Although she was knowing the answer, she didn't raised her hand during the class discussion."

# Classify grammatical acceptability
result = classifier(user_text)

print(result)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': 'LABEL_1', 'score': 0.788837730884552}]
